# 02a — Shared development validation protocol

Run after notebook 02 passes. This incremental milestone fixes **five stratified development folds** for both future models. No image from the test partition is opened and no model is trained. The manifest is read only to verify the split and select development rows.

Install first, from the project root:
```bat
venv\Scripts\python.exe -m pip install numpy pandas scikit-learn ipykernel
```

All logic is visible here. The final submission will inline this section after data preparation rather than depend on this notebook or a private module.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd().resolve()
METRICS = ROOT / 'outputs' / 'metrics'
REPORTS = ROOT / 'outputs' / 'reports'
MANIFEST = METRICS / 'data_split_manifest.csv'
LOCK = METRICS / 'split_lock.json'
STATUS = REPORTS / '02_split_status.json'
SEED, N_FOLDS = 42, 5
EXPECTED_CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled_in_scale', 'scratches']
def require(condition, message):
    if not condition:
        raise RuntimeError(message)
require(all(p.is_file() for p in [MANIFEST, LOCK, STATUS]), 'Run notebook 02 successfully first.')
lock = json.loads(LOCK.read_text(encoding='utf-8'))
status = json.loads(STATUS.read_text(encoding='utf-8'))
manifest_digest = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
require(status.get('status') == 'PASS', 'Notebook 02 has not passed its latest run.')
require(manifest_digest == lock.get('manifest_sha256') == status.get('manifest_sha256'),
        'Manifest, lock and PASS report disagree.')
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
require(manifest.path.is_unique, 'Duplicate paths in split manifest.')
development = manifest.loc[manifest.split.eq('development')].sort_values('path').reset_index(drop=True)
test_paths = set(manifest.loc[manifest.split.eq('test'), 'path'])
require(len(development) == 1439 and set(development.class_name) == set(EXPECTED_CLASSES),
        'Unexpected development partition.')
require(development.sha256.is_unique and development.pixel_sha256.is_unique, 'Development duplicates found.')
# Verify only development image bytes. Do not open test images during this stage.
for row in development.itertuples():
    path = (ROOT / row.path).resolve()
    require(path.is_relative_to(ROOT / 'data' / 'NEU-CLS'), f'Unsafe source path: {row.path}')
    require(hashlib.sha256(path.read_bytes()).hexdigest() == row.sha256, f'Changed development file: {row.path}')
print('Verified development data and split acceptance. Python:', sys.executable)

Verified development data and split acceptance. Python: C:\Scripts\industrial-defect-intelligence\venv\Scripts\python.exe


## Fixed folds for comparable experiments

Each row receives one `validation_fold` from 0 to 4. For fold *k*, train on all other development rows and validate on rows labelled *k*. Each model must use the same assignment. Five-fold validation means around **1,151 training / 288 validation images per fold**, not the full-dataset fold sizes discussed before duplicate removal and hold-out reservation.

The fold table is locked to the split-manifest checksum and its own checksum. Reruns verify and reuse it. Changes require review, not silent regeneration.

In [2]:
FOLDS_PATH = METRICS / 'development_cv_folds.csv'
FOLDS_LOCK_PATH = METRICS / 'development_cv_lock.json'
folds = development[['path', 'class_name', 'sha256', 'pixel_sha256']].copy()
if FOLDS_LOCK_PATH.exists():
    fold_lock = json.loads(FOLDS_LOCK_PATH.read_text(encoding='utf-8'))
    require(fold_lock['split_manifest_sha256'] == manifest_digest and
            fold_lock['seed'] == SEED and fold_lock['n_folds'] == N_FOLDS, 'Fold lock policy mismatch.')
    require(FOLDS_PATH.exists() and hashlib.sha256(FOLDS_PATH.read_bytes()).hexdigest() == fold_lock['folds_sha256'],
            'Fold table changed or missing.')
    saved = pd.read_csv(FOLDS_PATH)
    require(saved.path.is_unique and set(saved.path) == set(folds.path), 'Fold path mismatch.')
    folds['validation_fold'] = folds.path.map(saved.set_index('path').validation_fold)
else:
    require(not FOLDS_PATH.exists(), 'Review existing unlocked fold table before proceeding.')
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds['validation_fold'] = -1
    for fold, (_, validation_indices) in enumerate(cv.split(folds.path, folds.class_name)):
        folds.loc[validation_indices, 'validation_fold'] = fold

require(set(folds.validation_fold) == set(range(N_FOLDS)), 'Incomplete fold assignment.')
summary=[]
for fold in range(N_FOLDS):
    train = folds.loc[folds.validation_fold.ne(fold)]
    valid = folds.loc[folds.validation_fold.eq(fold)]
    require(set(train.class_name) == set(valid.class_name) == set(EXPECTED_CLASSES), 'Missing fold class.')
    for key in ['path', 'sha256', 'pixel_sha256']:
        require(not (set(train[key]) & set(valid[key])), f'Fold {fold} leaks on {key}.')
    require(not (set(train.path) | set(valid.path)) & test_paths, 'Test row entered development CV.')
    require(len(train) + len(valid) == 1439, 'Fold does not cover development pool.')
    summary.append({'fold': fold, 'train_images': len(train), 'validation_images': len(valid)})
balance = pd.crosstab(folds.class_name, folds.validation_fold).reindex(EXPECTED_CLASSES)
require((balance.max(axis=1) - balance.min(axis=1)).le(1).all(), 'Fold class imbalance.')
payload = folds.to_csv(index=False, lineterminator='\n').encode('utf-8')
digest = hashlib.sha256(payload).hexdigest()
if FOLDS_LOCK_PATH.exists():
    require(digest == fold_lock['folds_sha256'], 'Reconstructed folds disagree with saved lock.')
else:
    FOLDS_PATH.write_bytes(payload)
    FOLDS_LOCK_PATH.write_text(json.dumps({'schema_version': 1, 'split_manifest_sha256': manifest_digest,
        'folds_sha256': digest, 'seed': SEED, 'n_folds': N_FOLDS}, indent=2), encoding='utf-8')
pd.DataFrame(summary).to_csv(METRICS / 'development_cv_summary.csv', index=False)
balance.to_csv(METRICS / 'development_cv_class_counts.csv')
display(pd.DataFrame(summary))
display(balance)
(REPORTS / '02a_validation_status.json').write_text(json.dumps({'status': 'PASS',
    'split_manifest_sha256': manifest_digest, 'folds_sha256': digest,
    'n_folds': N_FOLDS, 'models_trained': 0}, indent=2), encoding='utf-8')
print('PASS: five shared development folds saved; no test images opened; no models trained.')

,fold,train_images,validation_images
0,0,1151,288
1,1,1151,288
2,2,1151,288
3,3,1151,288
4,4,1152,287


validation_fold,0,1,2,3,4
class_name,,,,,
crazing,48,48,48,48,48
inclusion,48,48,48,48,48
patches,48,48,48,48,47
pitted_surface,48,48,48,48,48
rolled_in_scale,48,48,48,48,48
scratches,48,48,48,48,48


PASS: five shared development folds saved; no test images opened; no models trained.


## Modelling decisions to carry forward

1. **HOG–SVM:** grayscale conversion and deterministic per-image HOG extraction; fit `StandardScaler` and the SVM inside each training fold. Use a bounded search and macro F1 for selection, recording all tried settings and timing. Preprocessing choices chosen by validation are hyperparameters too.
2. **Compact CNN:** same development folds; fresh weights per fold; training-only augmentation. Estimate any dataset-level normalisation on that fold's training data. Select a small, declared set of learning rates/dropout settings. Early stopping and epoch selection use development validation only. Record CPU/GPU, seeds and deterministic settings.
3. **Evaluation:** record accuracy, macro precision, macro recall and macro F1 plus per-class metrics. CV scores used to select a configuration are selection evidence, not an unbiased final performance estimate. Freeze both configurations and CNN epoch policy before one final held-out comparison.
4. **Analysis:** define brightness/contrast/sharpness subgroup thresholds on development data. Fix perturbation types/severities before final test analysis. Distinguish natural image-condition diagnostics from synthetic robustness. No demographic fairness claims. Show CNN Grad-CAM examples including errors; describe HOG plots as representation visualisations.
5. **Efficiency and deployment:** measure training time, feature-extraction and prediction time, model size, feature dimensions and CNN parameter count. Discuss human escalation and generalisation limits; actual deployment is not required.

Next notebook: `03_classical_hog_svm.ipynb`. This protocol milestone intentionally ends before model fitting so its saved folds and methodological choices can be inspected independently.